## Ingesting fact_order_items into bronze layer ##



--- Used structure streaming function and autloader to process the data into bronze layer \
---checkpoint is defined to know the saved data point - the point to identify from where to start the ingestion if the process get crashed

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *


In [0]:
dbutils.widgets.text("catalog_name","ecommerce","catalog_name")
dbutils.widgets.text("storage_account_name","abiecommerceadlsdev","storage_account_name")
dbutils.widgets.text("container_name","ecomm-raw-data","container_name")

In [0]:
dbutils.widgets.get("catalog_name"), dbutils.widgets.get("storage_account_name"), dbutils.widgets.get("container_name")



In [0]:

#adls_path
adls_path = f"abfss://{dbutils.widgets.get('container_name')}@{dbutils.widgets.get('storage_account_name')}.dfs.core.windows.net/order_items/landing/"
#checkpoint path
bronze_checkpoint_path = f"abfss://{dbutils.widgets.get('container_name')}@{dbutils.widgets.get('storage_account_name')}.dfs.core.windows.net/checkpoints/bronze/fact_order_items/"


- cloudFiles indicates that it is a streaming data from cloud \
- cloudFile format is csv \
- schema EvolutionMode is Rescue - which means the schema of the data is maintained by having extra columns in json format \
- so rescueddatacolumn is _rescued_data \ 
- loading file from adls_path \
- added columns like source and ingested time \




In [0]:
spark.readStream \
 .format("cloudFiles") \
 .option("cloudFiles.format", "csv")  \
 .option("cloudFiles.schemaLocation", bronze_checkpoint_path) \
 .option("cloudFiles.schemaEvolutionMode", "rescue") \
 .option("header", "true") \
 .option("cloudFiles.inferColumnTypes", "true") \
 .option("rescuedDataColumn", "_rescued_data") \
 .option("cloudFiles.includeExistingFiles", "true")  \
 .option("pathGlobFilter", "*.csv") \
 .load(adls_path) \
 .withColumn("ingest_timestamp", F.current_timestamp()) \
 .withColumn("source_file", F.col("_metadata.file_path")) \
 .writeStream \
 .outputMode("append") \
 .option("checkpointLocation", bronze_checkpoint_path) \
 .trigger(availableNow=True) \
 .toTable(f"{dbutils.widgets.get('catalog_name')}.bronze.brz_order_items") \
 .awaitTermination()